# Samaritan — GRPO on verifiable rewards

Trains the 4B student with **reinforcement learning against the harness's own
grader**. The SFT run taught the teacher's *style*: same answers in 27% fewer
tokens, which at a fixed budget read as +20pp — but lift the budget and the
base recovered 25 of 26 failures. Imitation bought efficiency, not capability.
GRPO rewards **being right**, so the model can find solution paths no teacher
demonstrated.

**Runtime → Change runtime type → A100 (or L4), then Run all.** Upload
`grpo-bundle.tar.gz` (~68 KB) when cell 1 asks.

### Why this is set up the way it is

- **The reward is the same Rust grader every eval in this project used.** A
  Python reimplementation would be a second oracle that can silently disagree
  with the measurement — which is how a model gets optimised toward the wrong
  target without anyone noticing.
- **Problems are generated (W7), so they are post-cutoff by construction.** The
  model cannot be rewarded for recall.
- **Difficulty 3, and a training seed no eval has used.** Train/test separation
  by construction rather than by trusting a decontamination filter.

### The one setting that decides whether this works

GRPO's advantage is computed **purely from spread within a group** of rollouts
on the same prompt. A group that is all-right or all-wrong contributes exactly
zero gradient — and a run of nothing but flat groups looks, from the outside,
identical to a healthy one.

So `--max-completion` is load-bearing. Measured on the d3 baseline eval:

| budget | replies that finished |
|---|---|
| 2048 | **0 / 13** |
| 4096 | 3 / 13 |
| 6144 | 9 / 13 |

Median need is **5,636 tokens**. At the old 2048 default every rollout would be
cut off, every reward would be 0, and the run would train nothing for hours of
paid GPU. Hence 6144 below — and the trainer now says so out loud if the first
ten groups come back unanimous.

## 1. Upload the bundle

In [ ]:
import os, glob

if not glob.glob('grpo-bundle.tar.gz'):
    from google.colab import files
    print('Select grpo-bundle.tar.gz (it is in your models folder)...')
    files.upload()
assert os.path.exists('grpo-bundle.tar.gz'), 'bundle not uploaded'
!rm -rf samaritan && mkdir -p samaritan && tar -xzf grpo-bundle.tar.gz -C samaritan
print(sorted(os.listdir('samaritan')))

## 2. Rust, then build the grader

The bundle carries three crates rather than the whole workspace — the full one
pulls a TLS stack and a bundled SQLite that have nothing to do with grading and
turn a 40-second build into a multi-minute one that can fail on something
unrelated. `--release` matters too: this binary is invoked once per batch for
the entire run.

In [ ]:
import os, glob, subprocess, time, json
t0 = time.time()
if not os.path.exists(os.path.expanduser('~/.cargo/bin/cargo')):
    subprocess.run('curl -sSf https://sh.rustup.rs | sh -s -- -y'
                   ' --profile minimal -q', shell=True, check=True)
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']

!cd samaritan && cargo build --release -q -p samaritan-corpus --example grade_batch
!cd samaritan && cargo build --release -q -p samaritan-curriculum --example generate
# cargo also emits a hash-suffixed copy; prefer the plain name, but fall
# back rather than failing on a layout difference.
def built(name):
    base = f'samaritan/target/release/examples/{name}'
    hits = [base] if os.path.exists(base) else sorted(glob.glob(base + '-*'))
    hits = [h for h in hits if not h.endswith('.d')]
    assert hits, f'{name} did not build'
    return os.path.abspath(hits[0])

GRADER = built('grade_batch')
GENERATE = built('generate')
print(f'built in {time.time()-t0:.0f}s')

# Prove the reward works BEFORE spending GPU time. A reward function that
# silently returns 0 trains the model to do nothing, slowly and expensively.
probe = '\n'.join([
    json.dumps({'given': '<think>x</think>\nAnswer: 42', 'answer': '42', 'answer_kind': 'exactMatch'}),
    json.dumps({'given': '<think>x</think>\nAnswer: 43', 'answer': '42', 'answer_kind': 'exactMatch'}),
])
out = subprocess.run([GRADER], input=probe, capture_output=True, text=True)
verdicts = [json.loads(l)['correct'] for l in out.stdout.splitlines() if l.strip()]
assert verdicts == [True, False], f'grader is wrong: {verdicts} / {out.stderr[:400]}'
print('grader verified:', verdicts)

# Also imports the trainer, so a syntax or import error surfaces here
# rather than after the model has loaded.
!cd samaritan && python training/test_reward_health.py | tail -2

## 3. Generate the training curriculum

Seed 1001 — **not** 777/883/884, which are eval seeds. Same generator, disjoint
problems, so there is no path by which training touches the measurement set.

In [ ]:
os.environ['SEED'] = '1001'           # NOT an eval seed
os.environ['DIFFICULTY'] = '3'        # the ~83% band, where reward has spread
os.environ['COUNT'] = '400'
os.environ['SPLIT_LABEL'] = 'train'
os.environ['OUT'] = '/content/grpo-train-d3.jsonl'
!{GENERATE}

import collections
rows = [json.loads(l) for l in open('/content/grpo-train-d3.jsonl') if l.strip()]
fam = collections.Counter(r['id'].split('-')[1] for r in rows)
print(f'\n{len(rows)} problems across {len(fam)} families')
print(dict(fam))

## 4. Install the training stack

`GRPOTrainer` needs `trl>=0.14`; the project's pin is `>=0.9`, which predates it.

In [ ]:
!pip -q install unsloth 'trl>=0.14' peft datasets 2>&1 | tail -3
import trl; print('trl', trl.__version__)

## 5. Train

Roughly 2–3.5 h for 200 steps on an A100 (~11–19 units at 5.3/h). Checkpoints
land every 25 steps, because Colab preempts.

**Watch `reward`, not `loss`.** Around step 10 the trainer prints a reward-health
line. If it reports no spread, stop — it will also tell you *which* problem it
is, since the two causes need opposite fixes:

- **few rollouts emitted an `Answer:` line** → they are being cut off. Raise
  `--max-completion`.
- **rollouts finished but agreed anyway** → problems all too easy or all too
  hard. Regenerate at a different `DIFFICULTY`.

Neither is fixed by training longer.

In [ ]:
!cd samaritan && python training/grpo_reasoning.py /content/grpo-train-d3.jsonl \
    --base-model Qwen/Qwen3-4B-Thinking-2507 \
    --output /content/adapters/reasoning-grpo \
    --grader {GRADER} \
    --max-seq-len 8192 \
    --max-completion 6144 \
    --generations 8 \
    --steps 200

### Optional: start from the SFT student instead

SFT-then-RL is the order DeepSeek-R1 used, and here it has a second benefit —
the student is 27% more concise, so more rollouts finish inside the budget and
more groups come back mixed. Upload the SFT adapter directory first, then:

```
--init-adapter /content/adapters/reasoning-sft
```

It continues *that* LoRA rather than stacking a fresh random one on top. Worth a
second run once the from-base numbers are in — the two answer different
questions (*what does RL alone buy?* vs *what is the best model I can build?*).

## 6. Export and download

**Do this before the runtime dies.** Colab's disk is ephemeral, and an adapter
left here is an afternoon of units spent on nothing.

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    '/content/adapters/reasoning-grpo', max_seq_length=8192,
    load_in_4bit=True, dtype=None)
model.save_pretrained_gguf('/content/gguf_grpo', tokenizer,
                           quantization_method='q4_k_m')
!ls -la /content/gguf_grpo/*.gguf

In [ ]:
from google.colab import files
g = sorted(glob.glob('/content/gguf_grpo/*.gguf'))[0]
print('downloading', g)
files.download(g)

## Then, locally

Import as `student-grpo-q4km` so it cannot be confused with the SFT student, and
run all three against the **held-out** d3 set (seed 883 — the one this training
run never saw):

```powershell
powershell -ExecutionPolicy Bypass -File scripts/ab-eval.ps1 `
  -Dataset "$env:USERPROFILE\models\reasoning\generated-d3-s883.jsonl" `
  -Models base-q4km,student-v1-q4km,student-grpo-q4km `
  -Limit 40 -MaxTokens 6000 -Ctx 8192 -Tag "-grpo"
```

**The comparison that matters is GRPO-student vs SFT-student**, at a budget
generous enough that concision is not the binding constraint — otherwise this
measures the same efficiency gain the SFT run already bought, a second time.

Keep the per-family breakdown. If families move together, one adapter suffices.
If some climb while others regress, that is the evidence that per-domain
specialisation is the real bottleneck — which is the open question the deferred
hypernetwork idea turns on.